# Classification Models for Predicting Cluster Labels from Gene Presence-Absence

In [78]:
# Libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegressionCV


In [54]:
# Set file paths
repo_root = Path.cwd().parent
CLUSTER_FILE = repo_root / "analysis" / "reportree" / "ridom_partitions.tsv"
PRESENCE_ABSENCE_FILE = repo_root / "analysis" / "panaroo" / "merged" / "gene_presence_absence.Rtab"

In [55]:
# Load labels and features
clusters = pd.read_csv(CLUSTER_FILE, sep="\t", index_col=0)
presence_absence = pd.read_csv(PRESENCE_ABSENCE_FILE, sep="\t", index_col=0)


In [56]:
# For the presence-absence matrix - set samples to rows and features to columns
pa_transposed = presence_absence.transpose()
# For the clusters - we only want the columns for single-6x1.0, single-88x1.0, and single-501x1.0
target_clusters = clusters[["single-6x1.0", "single-88x1.0", "single-501x1.0"]]
# Merge the clusters and presence-absence data on the sample names
merged_data = pa_transposed.merge(target_clusters, left_index=True, right_index=True)

In [ ]:
# Count the number of clusters at the 501x1.0 level that have less than 10 samples
cluster_counts = merged_data["single-501x1.0"].value_counts()
print("Number of clusters with less than 10 samples at 501x1.0 level:")
print(cluster_counts[cluster_counts < 10].shape[0])

# Get the number of samples in clusters with less than 10 samples at the 501x1.0 level 
print("Number of samples in clusters with less than 10 samples at 501x1.0 level:")
print(cluster_counts[cluster_counts < 10].sum())

# Remove clusters with less than 10 samples at the 501x1.0 level
clusters_to_remove = cluster_counts[cluster_counts < 10].index
filtered_data = merged_data[~merged_data["single-501x1.0"].isin(clusters_to_remove)]



Number of clusters with less than 5 samples at 501x1.0 level:
189
Number of samples in clusters with less than 5 samples at 501x1.0 level:
282


## Train/validation/test splits

The first important step is to split the model into sets for training, validation, and testing. Here, we will use a 60/20/20 train/validation/test split. The models are trained on the training set, evaluated iteratively with the validation set, and only the final model is tested on the test set. Because of the extreme class imbalances in our cluster labels, we should use stratified rather than random splits, which ensure that the same proportion of each class is present in the training, validation, and test sets. To achieve this, we also remove all samples that belong to clusters with less than 10 members at the 501-allele single-linkage threshold, to avoid having not enough samples to have enough in the training/val set for 5-fold CV with one in each fold. 

In [67]:
# Separate features and labels
X = filtered_data.drop(columns=["single-6x1.0", "single-88x1.0", "single-501x1.0"])
y = filtered_data[["single-6x1.0", "single-88x1.0", "single-501x1.0"]]


In [68]:
# Split training/validation from testing data
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size = 0.2, random_state=7, stratify=filtered_data[["single-501x1.0"]])


In [ ]:
# Set any values in the labels that start with 'singleton' to NAs, since these are not true cluster labels and we aren't interested in predicting them
y_train_val = y_train_val.applymap(lambda x: x if not str(x).startswith("singleton") else pd.NA)

# Use the MultiLabelBinarizer to convert the multi-label format into a binary format for each class
mlb = MultiLabelBinarizer()
y_train_val = mlb.fit_transform(y_train_val)

In [73]:
# Remove zero-variance features from the training/validation data
selector = VarianceThreshold(threshold=0)
X_train_val_selected = selector.fit_transform(X_train_val)

In [74]:
# Before modeling - how many features do we have before and after removing zero-variance features?
print(f"Number of features before removing zero-variance features: {X_train_val.shape[1]}")
print(f"Number of features after removing zero-variance features: {X_train_val_selected.shape[1]}") 



Number of features before removing zero-variance features: 23338
Number of features after removing zero-variance features: 20333


In [ ]:
# Implement a Classifier Chain using Logistic Regression with cross-validation and L1 regularization
# Since the clusters are imbalanced, we can set the class_weight parameter to 'balanced' to help address this issue
# Also, since the clusters are hierarchical, we can set the order of the chain to reflect this hierarchy (i.e. single-501x1.0 -> single-88x1.0 -> single-6x1.0)

classifier_chain = ClassifierChain(
    LogisticRegressionCV(Cs=[1e-4, 1e-3, 1e-2, 1e-1, 1e0], l1_ratios=[1], solver='saga', class_weight='balanced'), 
    order=[2, 1, 0])
classifier_chain.fit(X_train_val_selected, y_train_val)

# Save the model to a file
import joblib
model_path = repo_root / "analysis" / "models"/ "classifier_chain_model.joblib"
joblib.dump(classifier_chain, model_path)


c:\Users\ashaa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1891: UserWarning: l1_ratios parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\ashaa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
# Evaluate the model performance on the test set after applying the same preprocessing steps (i.e. removing zero-variance features and converting labels to binary format)
X_test_selected = selector.transform(X_test)
y_test = y_test.applymap(lambda x: x if not str(x).startswith("singleton") else pd.NA)
y_test = mlb.transform(y_test)
test_score = classifier_chain.score(X_test_selected, y_test)
print(f"Test set accuracy: {test_score:.4f}")

In [ ]:
# Also evaluate the model using other metrics such as precision, recall, and F1-score for each class
from sklearn.metrics import classification_report
y_pred = classifier_chain.predict(X_test_selected)
print("Classification report:")
# Make sure the whole thing prints and doesn't get truncated - set the display options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(classification_report(y_test, y_pred, target_names=mlb.classes_)) 
# Also show the confusion matrix for each class
from sklearn.metrics import multilabel_confusion_matrix
conf_matrix = multilabel_confusion_matrix(y_test, y_pred)
for i, class_name in enumerate(mlb.classes_):
    print(f"Confusion matrix for class {class_name}:")
    print(conf_matrix[i])
